# 03 — Separation Demo: Conditioned vs Unconditioned

Load both `best_conditioned.pt` and `best_unconditioned.pt`, separate the same test tracks,
and compare results side-by-side with audio playback, spectrograms, and BSS-eval metrics.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
from src.utils.audio import load_audio
import IPython.display as ipd
from src.evaluation.bss import windowed_bss_eval

from src.data.manifests import load_manifest
from src.models.factory import build_model
from src.models.apply import apply_model
from src.inference.separate import create_tensor_for_segment
from src.utils.io import load_config

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Load both models

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

CONDITIONED_CKPT   = REPO_ROOT / "outputs" / "checkpoints" / "best_conditioned.pt"
UNCONDITIONED_CKPT = REPO_ROOT / "outputs" / "checkpoints" / "best_unconditioned.pt"

def load_model(config_path, checkpoint_path, device):
    config = load_config(config_path)
    model = build_model(config["model"]["name"], config["model"].get("kwargs", {}))
    if checkpoint_path.exists():
        payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state = payload.get("model_state_dict", payload)
        model.load_state_dict(state)
        print(f"Loaded: {checkpoint_path.name}")
    else:
        print(f"WARNING: {checkpoint_path} not found — using random weights")
    model.to(device).eval()
    return model, config

model_cond,   cfg_cond   = load_model(REPO_ROOT / "configs" / "conditioned.yaml",   CONDITIONED_CKPT,   device)
model_uncond, cfg_uncond = load_model(REPO_ROOT / "configs" / "unconditioned.yaml", UNCONDITIONED_CKPT, device)

print(f"\nConditioned   note_conditioning = {getattr(model_cond, 'note_conditioning', False)}")
print(f"Unconditioned note_conditioning = {getattr(model_uncond, 'note_conditioning', False)}")

Device: cpu
Loaded: best_conditioned.pt
Loaded: best_unconditioned.pt

Conditioned   note_conditioning = True
Unconditioned note_conditioning = False


## 2. Load test tracks from manifest

In [3]:
manifest = load_manifest(REPO_ROOT / cfg_cond["dataset"]["manifest"], resolve_root=REPO_ROOT)
test_entries = [e for e in manifest if e["split"] == cfg_cond["dataset"]["test_split"]]
print(f"Test tracks available: {len(test_entries)}")
for i, e in enumerate(test_entries):
    print(f"  [{i}] {e['track_name']}")

Test tracks available: 8
  [0] demo1
  [1] demo2
  [2] demo3
  [3] demo4
  [4] demo5
  [5] demo6
  [6] demo7
  [7] demo8


## 3. Separation helper

In [4]:
def separate_track(model, mix, entry, device):
    """Run separation. For conditioned models, loads and attaches notes."""
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std

    with torch.no_grad():
        if getattr(model, "note_conditioning", False):
            notes = create_tensor_for_segment(
                entry["notes_csv"],
                segment_start=0,
                segment_end=mix.shape[-1],
            )
            model_input = torch.cat((normalized, notes), dim=0)
            sources = apply_model(model, model_input[None], progress=False, device=device)[0]
        else:
            sources = apply_model(model, normalized[None], progress=False, device=device)[0]

    sources = sources * ref_std + ref_mean
    return sources[0].cpu(), sources[1].cpu()

## 4. Metrics helper

In [5]:
def compute_bss_metrics(g1_ref, g2_ref, g1_est, g2_est, sr):
    """Compute SDR, SIR, SAR for both sources."""
    min_len = min(g1_ref.shape[-1], g1_est.shape[-1])

    refs = np.stack([
        g1_ref[:, :min_len].T.numpy(),
        g2_ref[:, :min_len].T.numpy(),
    ], axis=0)  # (2, time, channels)

    ests = np.stack([
        g1_est[:, :min_len].T.numpy(),
        g2_est[:, :min_len].T.numpy(),
    ], axis=0)

    sdr, sir, isr, sar, _ = windowed_bss_eval(
        refs, ests,
        window=sr, hop=sr,
        compute_permutation=True,
    )

    return {
        "Guitar 1": {
            "SDR": float(np.nanmedian(sdr[0])),
            "SIR": float(np.nanmedian(sir[0])),
            "SAR": float(np.nanmedian(sar[0])),
        },
        "Guitar 2": {
            "SDR": float(np.nanmedian(sdr[1])),
            "SIR": float(np.nanmedian(sir[1])),
            "SAR": float(np.nanmedian(sar[1])),
        },
    }

## 5. Select and separate a test track

In [6]:
TRACK_IDX = 0  # <-- change this to test different tracks

entry = test_entries[TRACK_IDX]
print(f"Track: {entry['track_name']}")

mix, sr = load_audio(entry["mix"])
g1_ref, _ = load_audio(entry["sources"]["guitar1"])
g2_ref, _ = load_audio(entry["sources"]["guitar2"])
print(f"Duration: {mix.shape[-1]/sr:.2f}s  |  Sample rate: {sr} Hz")

print("\nSeparating with conditioned model...")
g1_cond, g2_cond = separate_track(model_cond, mix, entry, device)

print("Separating with unconditioned model...")
g1_uncond, g2_uncond = separate_track(model_uncond, mix, entry, device)

print("Done.")

Track: demo1
Duration: 34.50s  |  Sample rate: 44100 Hz

Separating with conditioned model...


ValueError: Invalid file path or buffer object type: <class 'NoneType'>

## 6. Metrics comparison

In [ ]:
metrics_cond   = compute_bss_metrics(g1_ref, g2_ref, g1_cond,   g2_cond,   sr)
metrics_uncond = compute_bss_metrics(g1_ref, g2_ref, g1_uncond, g2_uncond, sr)

print(f"{'':20s} {'Conditioned':>14s}  {'Unconditioned':>14s}")
print("-" * 52)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        v_c = metrics_cond[source][metric]
        v_u = metrics_uncond[source][metric]
        label = f"{source} {metric}"
        print(f"{label:20s} {v_c:+10.2f} dB   {v_u:+10.2f} dB")

## 7. Audio playback

In [ ]:
min_len = min(mix.shape[-1], g1_cond.shape[-1], g1_uncond.shape[-1])

print("=== Mix ===")
ipd.display(ipd.Audio(mix[:, :min_len].numpy(), rate=sr))

for guitar, ref, est_c, est_u in [
    ("Guitar 1", g1_ref, g1_cond, g1_uncond),
    ("Guitar 2", g2_ref, g2_cond, g2_uncond),
]:
    print(f"\n=== {guitar} — Reference ===")
    ipd.display(ipd.Audio(ref[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Conditioned ===")
    ipd.display(ipd.Audio(est_c[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Unconditioned ===")
    ipd.display(ipd.Audio(est_u[:, :min_len].numpy(), rate=sr))

## 8. Spectrogram comparison

In [ ]:
def plot_spectrogram(wav, sr, ax, title="", vmin=-60, vmax=0):
    mono = wav.mean(dim=0)[:min_len]
    spec = torch.stft(mono, n_fft=2048, hop_length=512,
                      window=torch.hann_window(2048), return_complex=True)
    mag_db = 20 * torch.log10(spec.abs().clamp(min=1e-8))
    ax.imshow(mag_db.numpy(), aspect="auto", origin="lower",
              extent=[0, min_len/sr, 0, sr/2], cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_ylim(0, 8000)
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True, sharey=True)

plot_spectrogram(g1_ref,    sr, axes[0, 0], "G1 — Reference")
plot_spectrogram(g1_cond,   sr, axes[0, 1], "G1 — Conditioned")
plot_spectrogram(g1_uncond, sr, axes[0, 2], "G1 — Unconditioned")
plot_spectrogram(g2_ref,    sr, axes[1, 0], "G2 — Reference")
plot_spectrogram(g2_cond,   sr, axes[1, 1], "G2 — Conditioned")
plot_spectrogram(g2_uncond, sr, axes[1, 2], "G2 — Unconditioned")

for ax in axes[-1]:
    ax.set_xlabel("Time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Freq (Hz)")

plt.suptitle(f"Separation — {entry['track_name']}", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Batch evaluation over all test tracks

In [ ]:
all_metrics = {"conditioned": [], "unconditioned": []}

for i, entry_i in enumerate(test_entries):
    print(f"[{i+1}/{len(test_entries)}] {entry_i['track_name']}", end=" ... ")
    mix_i, sr_i = load_audio(entry_i["mix"])
    g1r_i, _ = load_audio(entry_i["sources"]["guitar1"])
    g2r_i, _ = load_audio(entry_i["sources"]["guitar2"])

    try:
        g1c, g2c = separate_track(model_cond, mix_i, entry_i, device)
        m_c = compute_bss_metrics(g1r_i, g2r_i, g1c, g2c, sr_i)
        all_metrics["conditioned"].append(m_c)
    except Exception as exc:
        print(f"  cond FAILED: {exc}")
        all_metrics["conditioned"].append(None)

    try:
        g1u, g2u = separate_track(model_uncond, mix_i, entry_i, device)
        m_u = compute_bss_metrics(g1r_i, g2r_i, g1u, g2u, sr_i)
        all_metrics["unconditioned"].append(m_u)
    except Exception as exc:
        print(f"  uncond FAILED: {exc}")
        all_metrics["unconditioned"].append(None)

    print("ok")

print("\n=== Aggregate (median over test tracks) ===")
print(f"{'':20s} {'Conditioned':>14s}  {'Unconditioned':>14s}")
print("-" * 52)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        vals_c = [m[source][metric] for m in all_metrics["conditioned"] if m is not None]
        vals_u = [m[source][metric] for m in all_metrics["unconditioned"] if m is not None]
        med_c = np.nanmedian(vals_c) if vals_c else float("nan")
        med_u = np.nanmedian(vals_u) if vals_u else float("nan")
        label = f"{source} {metric}"
        print(f"{label:20s} {med_c:+10.2f} dB   {med_u:+10.2f} dB")